<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.7-scaling-ai-apps/notebooks/exercises-10_7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.7: Scaling AI Applications

*Module 10 · AI System Architecture & Production Deployment*

> Your AI app works for 10 users. Now 10,000 hit it at once. LLM calls take 2-30 seconds each, tying up resources far longer than typical web requests. This lesson builds scaling strategies purpose-built for AI workloads: request-aware auto-scaling, queue-based decoupling, and GPU scaling for self-hosted models.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-10.7.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Cloud Run Auto-Scaling — The Settings That Matter for AI — `scaling_config.sh`
2. Step 1 — Cloud Run Auto-Scaling — The Settings That Matter for AI — `cold_start_optimizer.py`
3. Step 2 — Request-Aware Load Balancing — Route Smart, Not Round-Robin — `load_balancer.py`
4. Step 3 — Queue-Based Scaling with Pub/Sub — Absorb Any Burst — `queue_scaling.py`
5. Step 4 — GPU Scaling on GKE — Self-Hosted Models at Scale — `gke_gpu_scaling.yaml`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai openai fastapi requests


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Cloud Run Auto-Scaling — The Settings That Matter for AI

Default Cloud Run settings are tuned for web apps, not LLM calls. Three settings to change.

**`scaling_config.sh`** · _Block 1 of 5_

!/bin/bash — ═══════════════════════════════════════════


In [ ]:
#!/bin/bash
# ═══════════════════════════════════════════
# CLOUD RUN SCALING FOR AI WORKLOADS
# Default settings are for web apps (50ms).
# LLM calls take 2-30s. Tune accordingly.
# ═══════════════════════════════════════════

# ── WRONG: Default settings ──
# gcloud run deploy ai-service \
#   --concurrency 80      ← 80 concurrent requests per instance
#   --min-instances 0     ← scale to zero (cold starts!)
#   --max-instances 100   ← way too many for LLM costs
#   --timeout 300         ← 5 min default

# ── RIGHT: AI-tuned settings ──
gcloud run deploy ai-service \
  --image gcr.io/PROJECT/ai-service:latest \
  --region asia-south1 \
  \
  # CONCURRENCY: how many requests per instance?
  # Web apps: 80. LLM apps: 4-10.
  # Each LLM call uses ~200MB RAM + blocks for 2-20s.
  # At concurrency=10: one instance handles 10 parallel LLM calls.
  --concurrency 10 \
  \
  # MIN INSTANCES: keep warm to avoid cold starts.
  # Cold start = 2-5s (download image + load Python + import libs).
  # min-instances=1: always one instance warm. Cost: ~$5/month.
  --min-instances 1 \
  \
  # MAX INSTANCES: cap to control costs.
  # Each instance can make ~10 LLM calls concurrently.
  # 10 instances × 10 concurrency = 100 parallel LLM calls.
  # Beyond this, you'll hit the LLM API rate limit anyway.
  --max-instances 10 \
  \
  # TIMEOUT: LLM calls can be slow. Allow 60s.
  --timeout 60 \
  \
  # RESOURCES: LLM processing needs more RAM.
  --memory 1Gi \
  --cpu 2 \
  \
  # CPU BOOST: extra CPU during cold start for faster import.
  --cpu-boost \
  \
  # EXECUTION ENVIRONMENT: gen2 for better cold starts.
  --execution-environment gen2


**`cold_start_optimizer.py`** · _Block 2 of 5_

COLD START OPTIMIZATION — Lazy-load heavy imports. Warm up connections.


In [ ]:
# =============================================
# COLD START OPTIMIZATION
# Lazy-load heavy imports. Warm up connections.
# =============================================

from fastapi import FastAPI
import os, time

app = FastAPI()

# ── Lazy-load heavy modules ──
# DON'T: import google.generativeai at module level (adds 1-2s to cold start)
# DO: import on first request, cache the client

_llm_client = None

def get_llm():
    """Lazy-load the LLM client on first request."""
    global _llm_client
    if _llm_client is None:
        import google.generativeai as genai
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        _llm_client = genai.GenerativeModel("gemini-2.0-flash")
    return _llm_client

# ── Startup: warm connection pool ──
@app.on_event("startup")
async def warmup():
    """Pre-warm on startup (runs during min-instance keep-alive)."""
    start = time.time()
    client = get_llm()
    # Make one cheap call to warm the connection
    try:
        client.generate_content("hi")
    except:
        pass
    print(f"Warmup complete in {time.time()-start:.1f}s")

@app.get("/health")
async def health():
    return {"status": "healthy", "llm_loaded": _llm_client is not None}

@app.post("/v1/chat")
async def chat(request: dict):
    model = get_llm()
    resp = model.generate_content(request.get("prompt", ""))
    return {"response": resp.text}


> **What just happened?** Three critical settings changed: concurrency=10 (down from 80 — each LLM call ties up a slot for 2-20s), min-instances=1 (keeps one instance warm, eliminates cold starts for ~$5/month), max-instances=10 (caps cost — 10 instances × 10 concurrency = 100 parallel LLM calls, usually hitting the API rate limit before needing more). Cold start optimization: lazy-load google.generativeai on first request (not at import time), warm the connection pool at startup. Result: first request goes from 4-5s cold start to sub-second.


### Step 2 · Request-Aware Load Balancing — Route Smart, Not Round-Robin

A 3-second Flash call and a 25-second Pro call shouldn't get the same instance pool.

**`load_balancer.py`** · _Block 3 of 5_

REQUEST-AWARE LOAD BALANCER — Route requests by estimated latency/cost.


In [ ]:
# =============================================
# REQUEST-AWARE LOAD BALANCER
# Route requests by estimated latency/cost.
# Separate pools for fast vs slow models.
# =============================================

from dataclasses import dataclass
import time
from collections import defaultdict

@dataclass
class ServicePool:
    name: str
    url: str
    max_concurrency: int
    current_load: int = 0
    avg_latency_ms: float = 0

class AILoadBalancer:
    """Route AI requests to the right service pool based on expected load."""
    
    MODEL_PROFILES = {
        "gemini-2.0-flash":  {"pool": "fast",  "avg_latency_s": 2,  "cost_per_1k": 0.01},
        "gemini-2.5-flash":  {"pool": "fast",  "avg_latency_s": 3,  "cost_per_1k": 0.015},
        "gemini-2.5-pro":    {"pool": "slow",  "avg_latency_s": 15, "cost_per_1k": 0.10},
    }
    
    def __init__(self):
        self.pools = {
            "fast": ServicePool("fast", "https://ai-fast.run.app", max_concurrency=50),
            "slow": ServicePool("slow", "https://ai-slow.run.app", max_concurrency=20),
        }
        self.latency_tracker: dict[str, list] = defaultdict(list)
    
    def route(self, model: str, estimated_tokens: int = 0) -> dict:
        """Select the best pool for this request."""
        profile = self.MODEL_PROFILES.get(model, self.MODEL_PROFILES["gemini-2.0-flash"])
        pool_name = profile["pool"]
        pool = self.pools[pool_name]
        
        # Check capacity
        if pool.current_load >= pool.max_concurrency:
            return {"pool": pool_name, "url": pool.url,
                    "action": "queue", "reason": "Pool at capacity"}
        
        pool.current_load += 1
        return {"pool": pool_name, "url": pool.url,
                "action": "route", "load": f"{pool.current_load}/{pool.max_concurrency}"}
    
    def release(self, model: str, latency_ms: float):
        """Release a slot and track latency."""
        profile = self.MODEL_PROFILES.get(model, {})
        pool = self.pools.get(profile.get("pool", "fast"))
        pool.current_load = max(0, pool.current_load - 1)
        
        # Track rolling average
        self.latency_tracker[model].append(latency_ms)
        if len(self.latency_tracker[model]) > 100:
            self.latency_tracker[model] = self.latency_tracker[model][-100:]
    
    def status(self):
        print("Pool Status:")
        for name, pool in self.pools.items():
            print(f"  {name:6s}: {pool.current_load}/{pool.max_concurrency} active")
        for model, latencies in self.latency_tracker.items():
            avg = sum(latencies) / len(latencies) if latencies else 0
            print(f"  {model}: avg {avg:.0f}ms ({len(latencies)} calls)")

# ── Demo ──
lb = AILoadBalancer()

print("Load balancing:\n")
for model in ["gemini-2.0-flash", "gemini-2.0-flash", "gemini-2.5-pro", "gemini-2.5-pro"]:
    result = lb.route(model)
    print(f"  {model:22s} → pool={result['pool']:5s} load={result.get('load','queued')}")
    lb.release(model, 2000 if "flash" in model else 15000)

lb.status()


> **What just happened?** AILoadBalancer separates traffic into two pools: fast (Flash models, 2-3s latency, 50 concurrent) and slow (Pro models, 15s latency, 20 concurrent). A 2-second Flash call doesn't compete for slots with a 25-second Pro call. When a pool is at capacity, the request is queued instead of rejected. Latency tracking enables adaptive pool sizing. Without this separation, one burst of Pro requests starves all Flash users.


### Step 3 · Queue-Based Scaling with Pub/Sub — Absorb Any Burst

Accept all requests instantly. Process them at a sustainable rate. Never drop a request.

**`queue_scaling.py`** · _Block 4 of 5_

QUEUE-BASED SCALING WITH PUB/SUB — Accept instantly → queue → process at rate.


In [ ]:
# =============================================
# QUEUE-BASED SCALING WITH PUB/SUB
# Accept instantly → queue → process at rate.
# =============================================

from fastapi import FastAPI, BackgroundTasks
from google.cloud import pubsub_v1, firestore
import json, uuid, os

app = FastAPI()

# ── Config ──
PROJECT = os.getenv("GCP_PROJECT", "your-project")
TOPIC = "ai-requests"

# ── API Server: accepts requests instantly ──

@app.post("/v1/chat/async")
async def submit_request(request: dict):
    """Accept request instantly. Return job_id for polling."""
    job_id = str(uuid.uuid4())[:8]
    
    # Publish to Pub/Sub (non-blocking, ~10ms)
    publisher = pubsub_v1.PublisherClient()
    topic_path = publisher.topic_path(PROJECT, TOPIC)
    
    message = json.dumps({
        "job_id": job_id,
        "prompt": request.get("prompt", ""),
        "model": request.get("model", "gemini-2.0-flash"),
    }).encode()
    
    publisher.publish(topic_path, message)
    
    # Return immediately — no LLM call here!
    return {"job_id": job_id, "status": "queued", "poll_url": f"/v1/result/{job_id}"}

@app.get("/v1/result/{job_id}")
async def get_result(job_id: str):
    """Poll for result."""
    db = firestore.Client()
    doc = db.document(f"ai_results/{job_id}").get()
    if doc.exists:
        return doc.to_dict()
    return {"job_id": job_id, "status": "processing"}

# ── Worker: processes from Pub/Sub ──
# This runs as a SEPARATE Cloud Run service
# with a Pub/Sub push subscription.

worker_app = FastAPI()

@worker_app.post("/process")
async def process_message(request: dict):
    """Process a single Pub/Sub message (LLM call happens HERE)."""
    import base64
    message = request.get("message", {})
    data = json.loads(base64.b64decode(message.get("data", "")))
    
    job_id = data["job_id"]
    
    # Make the LLM call (this is the slow part)
    import google.generativeai as genai
    genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    model = genai.GenerativeModel(data.get("model", "gemini-2.0-flash"))
    resp = model.generate_content(data["prompt"])
    
    # Store result in Firestore
    db = firestore.Client()
    db.document(f"ai_results/{job_id}").set({
        "job_id": job_id,
        "status": "completed",
        "response": resp.text,
        "model": data.get("model"),
    })
    
    return {"status": "processed"}

print("""
GCP Setup:

  # Create Pub/Sub topic
  gcloud pubsub topics create ai-requests

  # Deploy API server (fast, no LLM calls)
  gcloud run deploy ai-api \\
    --concurrency 200 --min-instances 1 --max-instances 5

  # Deploy worker (slow, makes LLM calls)
  gcloud run deploy ai-worker \\
    --concurrency 10 --min-instances 1 --max-instances 20

  # Create push subscription (Pub/Sub → worker)
  gcloud pubsub subscriptions create ai-worker-sub \\
    --topic ai-requests \\
    --push-endpoint https://ai-worker-xxx.run.app/process \\
    --ack-deadline 120

  Result:
    API server: handles 1,000 req/s (just publish to Pub/Sub)
    Workers: process at 200 LLM calls/s (20 instances × 10 concurrent)
    Burst: Pub/Sub queues absorb any spike (millions of messages)
""")


> **What just happened?** Two separate services: API server (fast — accepts requests, publishes to Pub/Sub, returns job_id in ~10ms, concurrency=200) and Worker (slow — receives Pub/Sub messages, makes LLM calls, stores results in Firestore, concurrency=10). The API server handles 1,000 req/s easily because it never calls the LLM. Pub/Sub absorbs any burst (millions of messages). Workers process at a sustainable rate. Clients poll /v1/result/{job_id} . This architecture handles traffic spikes that would crash a synchronous service.


### Step 4 · GPU Scaling on GKE — Self-Hosted Models at Scale

When API costs exceed self-hosting, run your own model on GPU nodes with auto-scaling.

**`gke_gpu_scaling.yaml`** · _Block 5 of 5_

═══════════════════════════════════════════ — GKE GPU SCALING FOR SELF-HOSTED LLMs


In [ ]:
# ═══════════════════════════════════════════
# GKE GPU SCALING FOR SELF-HOSTED LLMs
# vLLM serving + GPU node pool + HPA
# ═══════════════════════════════════════════

# ── Step 1: Create GPU node pool ──
# gcloud container node-pools create gpu-pool \
#   --cluster ai-cluster \
#   --zone asia-south1-a \
#   --machine-type g2-standard-8 \
#   --accelerator type=nvidia-l4,count=1 \
#   --num-nodes 1 \
#   --enable-autoscaling \
#   --min-nodes 0 \
#   --max-nodes 4

# ── Step 2: Deployment (vLLM serving) ──
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: llm-server
spec:
  replicas: 1
  selector:
    matchLabels:
      app: llm-server
  template:
    metadata:
      labels:
        app: llm-server
    spec:
      nodeSelector:
        cloud.google.com/gke-accelerator: nvidia-l4
      containers:
        - name: vllm
          image: vllm/vllm-openai:latest
          args:
            - "--model"
            - "google/gemma-2-9b-it"
            - "--max-model-len"
            - "8192"
            - "--gpu-memory-utilization"
            - "0.9"
          ports:
            - containerPort: 8000
          resources:
            limits:
              nvidia.com/gpu: "1"
              memory: "24Gi"
            requests:
              nvidia.com/gpu: "1"
              memory: "16Gi"
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 120
            periodSeconds: 10

# ── Step 3: Service ──
---
apiVersion: v1
kind: Service
metadata:
  name: llm-server-svc
spec:
  selector:
    app: llm-server
  ports:
    - port: 80
      targetPort: 8000
  type: ClusterIP

# ── Step 4: Horizontal Pod Autoscaler ──
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: llm-server-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: llm-server
  minReplicas: 1
  maxReplicas: 4
  metrics:
    # Scale on GPU utilization
    - type: Pods
      pods:
        metric:
          name: gpu_utilization
        target:
          type: AverageValue
          averageValue: "70"
    # Scale on request queue depth
    - type: Pods
      pods:
        metric:
          name: pending_requests
        target:
          type: AverageValue
          averageValue: "5"


> **What just happened?** Self-hosted LLM on GKE: a GPU node pool (NVIDIA L4, auto-scales 0-4 nodes), a vLLM deployment serving Gemma-2 9B with 90% GPU memory utilization, and a Horizontal Pod Autoscaler that scales on GPU utilization (>70% → add pods) and request queue depth (>5 pending → add pods). When to self-host: if you're spending >$5,000/month on API calls, an L4 GPU at ~$0.70/hour ($500/month) serving 100+ requests/second on a 9B model saves 90%. Below $5K/month, API calls are cheaper and simpler.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
